# Análisis Exploratorio de Datos (EDA)
**Dataset:** Siniestros Viales en Gran Bretaña (2021-2025) - STATS19

En este notebook se realiza la exploración formal del dataset limpio:
1. Carga de los datos procesados desde `data/processed/`.
2. Inspección univariada de la variable objetivo ($Y$: Severidad).
3. Distribución de las variables explicativas clave ($X$: Velocidad, Entorno y Vía).
4. Patrones temporales intradía ($T$: Horas del día).

In [ ]:
import pandas as pd
import plotly.express as px

DATA_PATH = "../data/processed/datos_colisiones_procesados.csv"

df = pd.read_csv(DATA_PATH)
df['fecha'] = pd.to_datetime(df['fecha'])

print(f"Total registros cargados: {len(df):,}")
df.head(3)

Total registros cargados: 513,740


,collision_index,collision_year,fecha,hora,dia_semana,severidad,es_grave_fatal,es_fatal,speed_limit,entorno,tipo_via,estado_calzada,clima,iluminacion
0,202517M102225,2025,2025-02-15,19,Sábado,Leve,0,0,30,Urbano,Calzada simple,Seco,Despejado,Noche (con alumbrado)
1,202417S111924,2024,2024-10-22,14,Martes,Grave,1,0,30,Urbano,Calzada simple,Seco,Despejado,Luz diurna
2,2025111687011,2025,2025-12-19,6,Viernes,Leve,0,0,30,Urbano,Calzada simple,Mojado / Húmedo,Lluvia,Noche (con alumbrado)


## 1. Distribución de la Variable Objetivo ($Y$)
Evaluamos el balance de clases de la severidad del impacto: `Fatal`, `Grave` y `Leve`.

In [14]:
resumen_y = df['severidad'].value_counts(normalize=True).reset_index()
resumen_y.columns = ['severidad', 'proporcion']
resumen_y['porcentaje'] = (resumen_y['proporcion'] * 100).round(2)

print(resumen_y)

fig_y = px.bar(
    resumen_y,
    x='severidad',
    y='porcentaje',
    color='severidad',
    category_orders={'severidad': ['Leve', 'Grave', 'Fatal']},
    color_discrete_map={'Fatal': '#e63946', 'Grave': '#f4a261', 'Leve': '#2a9d8f'},
    labels={'severidad': 'Severidad', 'porcentaje': 'Proporción (%)'},
    title="Distribución Porcentual por Severidad de la Colisión (Variable Y)"
)
fig_y.show()

  severidad  proporcion  porcentaje
0      Leve    0.757957       75.80
1     Grave    0.227343       22.73
2     Fatal    0.014700        1.47


> **Observación:** Existe un fuerte desbalance de clases. La categoría `Leve` concentra más del 70% del universo, mientras que los eventos `Fatal` representan menos del 2%. Por este motivo, el análisis posterior requiere normalización de tasas relativas.

## 2. Variables Explicativas de la Vía y Entorno ($X$)
Exploramos los límites de velocidad reglamentarios y la configuración de la via.

In [15]:

fig_vel = px.histogram(
    df,
    x='speed_limit',
    nbins=12,
    color_discrete_sequence=['#457b9d'],
    title="Distribución de Frecuencia por Limite de Velocidad (mph)",
    labels={'speed_limit': 'Límite de Velocidad (mph)', 'count': 'Frecuencia'}
)
fig_vel.show()

vias = df['tipo_via'].value_counts().reset_index()
vias.columns = ['tipo_via', 'conteo']

fig_vias = px.bar(
    vias,
    x='conteo',
    y='tipo_via',
    orientation='h',
    color_discrete_sequence=['#1d3557'],
    title="Colisiones según Tipo de Vía"
)
fig_vias.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_vias.show()

> **Observación:** El limite de velocidad mas frecuente es 30 mph (tipico de vias urbanas). Ademas, las colisiones ocurren predominantemente en calzadas simples (*single carriageway*), superando ampliamente a vías de doble calzada o rotondas.

## 3. Patrones Horarios e Iluminación ($T$ intradía y $X$ ambiental)
Identificación de los rangos horarios con mayor frecuencia de siniestros y su condicion luminica.

In [16]:
fig_hora = px.histogram(
    df.dropna(subset=['hora']),
    x='hora',
    nbins=24,
    color_discrete_sequence=['#2a9d8f'],
    title="Distribución de Colisiones por Hora del Día",
    labels={'hora': 'Hora (0-23)', 'count': 'Total Colisiones'}
)
fig_hora.show()

> **Conclusiones del EDA:**  
> - El volumen de siniestros está dominado por zonas urbanas de 30 mph en horarios punta (08:00 y 17:00-18:00 hrs).  
> - Pese al predominio de casos leves, los factores críticos para modelar la gravedad alta ($X \to Y$) serán analizados mediante tasas en el módulo analítico de Streamlit.